# 05 — Aggregations
**Goal:** Collapse arrays into summary numbers — totals, averages, extremes — and understand the `axis` parameter.

```
axis=None  → collapse entire array → scalar
axis=0     → collapse rows        → result per column
axis=1     → collapse columns     → result per row
```

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd

## 1. Basic aggregations on 1D arrays

In [ ]:
sessions = np.array([3200, 2800, 3500, 2600, 4100])

print('sum:    ', sessions.sum())
print('mean:   ', sessions.mean())
print('median: ', np.median(sessions))
print('std:    ', sessions.std().round(2))
print('var:    ', sessions.var().round(2))
print('min:    ', sessions.min())
print('max:    ', sessions.max())
print('range:  ', sessions.max() - sessions.min())
print('argmin: ', sessions.argmin())   # index of the minimum
print('argmax: ', sessions.argmax())   # index of the maximum

## 2. The `axis` parameter — the most important concept here

In [ ]:
# Funnel matrix: rows = days (3), cols = channels (5)
daily_sessions = np.array([
    [3200, 2800, 3500, 2600, 4100],   # day 1
    [3100, 2900, 3400, 2500, 4000],   # day 2
    [3300, 2700, 3600, 2700, 4200],   # day 3
])
channels = ['organic', 'paid', 'email', 'social', 'direct']

print('Shape:', daily_sessions.shape)   # (3, 5)

In [ ]:
# axis=0 → collapse ROWS → result has shape (5,) → one value per channel
channel_total = daily_sessions.sum(axis=0)
print('Total per channel:', channel_total)
for ch, val in zip(channels, channel_total):
    print(f'  {ch:8s}: {val:,}')

In [ ]:
# axis=1 → collapse COLUMNS → result has shape (3,) → one value per day
daily_total = daily_sessions.sum(axis=1)
print('Total per day:', daily_total)

In [ ]:
# axis=None (default) → global scalar
grand_total = daily_sessions.sum()
print('Grand total sessions:', grand_total)

## 3. keepdims — preserve shape for broadcasting

In [ ]:
# Without keepdims: mean per channel → shape (5,)
mean_flat = daily_sessions.mean(axis=0)
print('shape without keepdims:', mean_flat.shape)

# With keepdims: → shape (1, 5) → can broadcast back against (3, 5)
mean_kd = daily_sessions.mean(axis=0, keepdims=True)
print('shape with keepdims:   ', mean_kd.shape)

# Now we can compute deviation from channel mean:
deviation = daily_sessions - mean_kd
print('Deviation from channel daily mean:')
print(deviation)

## 4. Percentiles and quantiles

In [ ]:
sessions = np.array([3200, 2800, 3500, 2600, 4100, 2200, 4500, 3100, 2900, 3800])

print('p25 (Q1):', np.percentile(sessions, 25))
print('p50 (Q2):', np.percentile(sessions, 50))   # same as median
print('p75 (Q3):', np.percentile(sessions, 75))
print('p90:     ', np.percentile(sessions, 90))
print('IQR:     ', np.percentile(sessions, 75) - np.percentile(sessions, 25))

# Multiple percentiles in one call
p = np.percentile(sessions, [25, 50, 75, 90, 95])
print('\nAll at once:', p)

## 5. Cumulative functions

In [ ]:
daily_activations = np.array([27, 18, 14, 5, 5, 30, 20, 16, 4, 6])

# Running total — useful for cumulative conversion plots
print('cumsum:', np.cumsum(daily_activations))

# Cumulative max — useful for tracking all-time high
print('cummax:', np.maximum.accumulate(daily_activations))

# Day-over-day difference
print('diff:  ', np.diff(daily_activations))

## 6. Real example — channel summary from funnel data

In [ ]:
df = pd.read_csv('data/raw/funnel_data.csv')
channels = ['organic', 'paid', 'email', 'social', 'direct']

for ch in channels:
    sub = df[df['channel'] == ch]
    s   = sub['visita_landing'].to_numpy(dtype=float)
    a   = sub['activacion_tarjeta'].to_numpy(dtype=float)
    cvr = a / s * 100
    print(f'{ch:8s} | sessions mean={s.mean():,.0f} | CVR mean={cvr.mean():.2f}% | CVR p90={np.percentile(cvr, 90):.2f}%')

## Summary
| Function | What it does |
|---|---|
| `.sum()` / `.mean()` / `.std()` | Scalar or along axis |
| `.min()` / `.max()` | Extremes |
| `.argmin()` / `.argmax()` | Index of extreme |
| `np.median(arr)` | Median |
| `np.percentile(arr, q)` | Percentile(s) |
| `np.cumsum(arr)` | Running total |
| `np.diff(arr)` | Consecutive differences |
| `axis=0` | Collapse rows → per-column result |
| `axis=1` | Collapse columns → per-row result |
| `keepdims=True` | Preserve shape for broadcasting |

**Next:** `06_reshaping.ipynb` — reshape, transpose, stack, and concatenate.